# Databricks Lakehouse — Architecture & Compute

Introduction to the Databricks Lakehouse platform: architecture layers, compute types (all-purpose cluster, job cluster, SQL Warehouse), and notebook best practices.

| Training Block | Duration | Type |
|---|---|---|
| Lakehouse Architecture & Compute — Demo | 20 min | Demo |

**Prerequisites:** none — this module opens Day 2

## Learning Objectives

After completing this module you will be able to:

- **Explain** the Databricks Lakehouse layers (Bronze/Silver/Gold over Delta Lake)
- **Distinguish** three compute types: All-purpose Cluster, Job Cluster, SQL Warehouse
- **Choose** the right compute for the right workload
- **Connect** a SQL Warehouse and run interactive SQL analytics
- **Describe** how Power BI / BI tools connect to Databricks via SQL Warehouse
- **Apply** notebook best practices (structure, `%run`, widgets, `%pip`)

## Setup

In [0]:
%run ../../setup/00_setup

## 1. Databricks Lakehouse — Overview

The Lakehouse combines the **flexibility of a data lake** (raw files, scalability) with the **reliability of a data warehouse** (ACID, schemas, SQL queries).

![alt text](../../../assets/images/cf6a156ef8384d6b9473300b2160e11f.webp)

**Key principle:** data lives in one place (Delta Lake on object storage); compute is a separate, pluggable layer.

## 2. Compute Types in Databricks

### All-purpose Cluster (Shared / Interactive)

| Property | Value |
|---|---|
| Start | Manual (or auto-start) |
| Lifetime | Long-running — stays up until stopped |
| Users | Multiple simultaneous (shared cluster) |
| Cost | Billed for the entire uptime |
| Typical use | Data exploration, development, debugging |

```python
# Working on an all-purpose cluster — just use a notebook
spark.read.table("bronze.orders").display()
```

### Job Cluster (Automated / Ephemeral)

| Property | Value |
|---|---|
| Start | Automatic — spun up for the job only |
| Lifetime | Short-lived — terminated when the job finishes |
| Users | 1 job = 1 cluster |
| Cost | Billed only during job execution |
| Typical use | Production pipelines, DLT, scheduled jobs |

> **Best practice:** always run production pipelines on a Job Cluster — cheaper and isolated.

### SQL Warehouse (for analysts and BI)

| Property | Value |
|---|---|
| Start | Automatic (auto-start on first query) |
| Protocol | JDBC/ODBC — compatible with any BI tool |
| Users | Multiple simultaneous (auto-scale) |
| Cost | Billed only during active use (auto-stop on idle) |
| Typical use | Power BI, Tableau, Looker, Databricks SQL Editor |

> **Key difference:** SQL Warehouse is **optimised for SQL**, not PySpark. It uses the Photon engine by default.

### Serverless Compute

A variant of both all-purpose and SQL Warehouse — no infrastructure management, instant start, Databricks manages the entire fleet. Higher per-DBU cost, but zero operational overhead.

## 3. When to Use Which Compute?

| Task | Compute | Why |
|---|---|---|
| Data exploration in a notebook | **All-purpose Cluster** | Interactive, PySpark, iterative |
| Production ETL pipeline | **Job Cluster** | Ephemeral, isolated, cheaper |
| Lakeflow / DLT pipeline | **Job Cluster** (auto) | DLT manages it automatically |
| SQL query for analysis | **SQL Warehouse** | Optimised SQL, auto-scale |
| Power BI / Tableau dashboard | **SQL Warehouse** | JDBC/ODBC, certified connector |
| Ad-hoc analyst query | **SQL Warehouse** | Fast start, no Spark overhead |
| ML training | **All-purpose / Job** | GPU support, MLflow integration |

> **Simple rule:**
> - **writing code (Python / Scala / PySpark)** → cluster
> - **doing SQL analytics or BI** → SQL Warehouse

## 4. SQL Warehouse — Demo

### How It Works

![alt text](../../../assets/images/fe9202f8a7024d09a1efab3bd433c425.webp)

### Step 1 — Create a SQL Warehouse in the UI

1. Click **SQL Warehouses** in the left panel
2. Click **Create SQL Warehouse**
3. Choose size: `Small` (demo) or `Medium` (production)
4. Set **Auto Stop**: e.g. 10 minutes
5. Click **Create**

> **TRAINER:** show warehouse creation (or an existing one) in the UI

### Step 2 — Run a SQL Query (Databricks SQL Editor)

In [0]:
%sql
-- Demo: SQL Warehouse query — run this in Databricks SQL Editor (not in the notebook)
-- Example query using Databricks sample dataset
SELECT
  pickup_zip,
  COUNT(*) AS trip_count,
  ROUND(SUM(fare_amount), 2) AS total_fare
FROM samples.nyctaxi.trips
WHERE tpep_pickup_datetime >= '2016-01-01'
GROUP BY pickup_zip
ORDER BY total_fare DESC
LIMIT 20;

> **Photon Engine:** SQL Warehouse uses **Photon** — a native vectorised SQL engine written in C++.
> On SQL/BI workloads it delivers **2–4x** faster response times compared to standard Spark.

## 5. Power BI — Live Direct Query Demo

**Goal:** generate a Delta table that receives 1 new row every second, then connect Power BI in **Direct Query** mode with **auto page-refresh** — the report updates automatically without pressing Refresh.

```
Spark Structured Streaming (rate source)
        | 1 row/second
        v
  Delta Table: gold.live_orders
        |
        |  JDBC / native connector (Direct Query)
        v
  SQL Warehouse  →  Power BI (auto page-refresh 2 s)
```

### What we'll build

| Step | What | Tool |
|---|---|---|
| 1 | Create target Delta table | PySpark |
| 2 | Start streaming generator (1 row/s) | Structured Streaming |
| 3 | Verify data is flowing | Spark SQL |
| 4 | Connect Power BI — Direct Query + auto-refresh | Power BI Desktop |
| 5 | Stop the stream | PySpark |

### Step 1 — Create the target Delta table

In [0]:
# Create (or recreate) the live_orders table in the Gold schema
spark.sql(f"""
    CREATE OR REPLACE TABLE  {CATALOG}.{GOLD_SCHEMA}.live_orders (
        event_time   TIMESTAMP,
        order_id     BIGINT,
        customer_id  INT,
        category     STRING,
        order_amount DECIMAL(10,2),
        store_region STRING
    )
    USING DELTA
""")

spark.sql(f"TRUNCATE TABLE {CATALOG}.{GOLD_SCHEMA}.live_orders")
print(f"Table ready: {CATALOG}.{GOLD_SCHEMA}.live_orders")

### Step 2 — Start the streaming generator (1 row / second)

Uses Spark **Structured Streaming** with the built-in `rate` source — no external data needed.
Each micro-batch appends exactly 1 row to the Delta table.

In [0]:
from pyspark.sql import functions as F

CATEGORIES   = ["Electronics", "Clothing", "Food", "Books", "Sports"]
REGIONS      = ["North", "South", "East", "West", "Central"]

# ── rate source: emits (timestamp, value) at 1 row/second ──
raw_stream = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 1)
        .load()
)

# ── enrich with business fields ──
live_stream = raw_stream.select(
    F.col("timestamp").alias("event_time"),
    F.col("value").alias("order_id"),
    (F.col("value") % 50 + 1).cast("int").alias("customer_id"),
    F.element_at(
        F.array(*[F.lit(c) for c in CATEGORIES]),
        (F.col("value") % len(CATEGORIES) + 1).cast("int")
    ).alias("category"),
    (F.rand() * 490 + 10).cast("decimal(10,2)").alias("order_amount"),
    F.element_at(
        F.array(*[F.lit(r) for r in REGIONS]),
        (F.col("value") % len(REGIONS) + 1).cast("int")
    ).alias("store_region"),
)

# ── write to Delta table ──
live_query = (
    live_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"/tmp/checkpoints/live_orders_{CATALOG}")
        .toTable(f"{CATALOG}.{GOLD_SCHEMA}.live_orders")
)

print(f"Stream running — writing 1 row/s to {CATALOG}.{GOLD_SCHEMA}.live_orders")
print(f"Stream ID: {live_query.id}")

### Step 3 — Verify data is flowing

Run this cell a few times — the row count should increase by ~1 each second.

In [0]:
import time
time.sleep(5)   # wait for a few rows to land

spark.sql(f"""
    SELECT
        COUNT(*)                   AS total_rows,
        MAX(event_time)            AS latest_event,
        ROUND(SUM(order_amount),2) AS total_revenue
    FROM {CATALOG}.{GOLD_SCHEMA}.live_orders
""").display()

print("--- latest 5 rows ---")
spark.sql(f"""
    SELECT *
    FROM   {CATALOG}.{GOLD_SCHEMA}.live_orders
    ORDER BY event_time DESC
    LIMIT 5
""").display()

### Step 4 — Connect Power BI (Direct Query + Auto Page Refresh)

#### 4a — Get SQL Warehouse connection details

1. Open **SQL Warehouses** in the Databricks sidebar
2. Click your warehouse → **Connection details** tab
3. Copy **Server hostname** and **HTTP Path**

#### 4b — Connect Power BI Desktop

1. **Get Data** → search for **Databricks** → select the certified connector
2. Paste **Server hostname** and **HTTP Path** → click **OK**
3. Authentication: choose **Personal Access Token** (PAT) → paste your token
4. In the Navigator, expand your catalog: `{CATALOG}` → `gold` → tick **live_orders** → **Load**
5. In the **Connection Settings** dialog choose **DirectQuery** → click **OK**

> **DirectQuery** means every chart refresh sends a live SQL query to the SQL Warehouse — no data is cached in Power BI.

#### 4c — Build a simple live chart

1. Insert a **Card** visual → drag `total_rows = COUNT(order_id)` into it
2. Insert a **Bar Chart** → X-axis: `category`, Y-axis: `SUM(order_amount)`
3. Insert a **Table** visual → columns: `event_time`, `customer_id`, `category`, `order_amount`

#### 4d — Enable Auto Page Refresh

> Requires **Power BI Desktop** ≥ March 2020 and a workspace on **Premium / Fabric capacity** for intervals below 30 min. For demo purposes use a local Power BI Desktop connected to a SQL Warehouse.

1. Click anywhere on the **canvas** (deselect all visuals)
2. In the **Visualizations** pane open **Page** settings (canvas icon)
3. Scroll to **Page refresh** → toggle **On**
4. Set interval: **2 seconds** (minimum allowed depends on capacity)
5. Save and publish — the page will re-query the SQL Warehouse every 2 s

![alt text](../../../assets/images/495258d03c654f39843544075e009eeb.webp)

| Setting | Value for demo |
|---|---|
| Connection mode | DirectQuery |
| Auto page refresh interval | 2 seconds |
| Refresh trigger | Fixed interval |
| SQL Warehouse auto-stop | Disable during demo (or set 30 min) |


### Step 5 — Stop the stream (cleanup)

In [0]:
# Run this cell after the demo to stop the streaming query
live_query.stop()
print("Stream stopped.")
print(f"Final row count: {spark.table(f'{CATALOG}.{GOLD_SCHEMA}.live_orders').count()}")

## Summary

| Topic | Key takeaway |
|---|---|
| Lakehouse | Delta Lake as one storage layer; compute is a separate layer |
| All-purpose Cluster | Interactive notebooks, development, exploration |
| Job Cluster | Production, pipelines, cheaper, isolated |
| SQL Warehouse | SQL analytics, BI tools (Power BI / Tableau), Photon engine |
| Power BI | Direct Query via JDBC / native connector — live data |

-> [02 — Delta Advanced](02_delta_advanced_demo.ipynb) | **[README](../../../README.md)**